In [ ]:
from abc import ABC,abstractmethod

# Common class for all payment methods
class PaymentMethod(ABC):
    @abstractmethod
    def get_details(self):
        pass

    @abstractmethod
    def pay(self,amount):
        pass

# Payment using Razorpay card
class RazorpayCardPayment(PaymentMethod):
    def __init__(self,card_number):
        self.card_number=card_number

    def get_details(self):
        return "Razorpay Card ending with "+self.card_number[-4:]

    def pay(self,amount):
        print("Payment of Rs.",amount,"made using",self.get_details())
        return True

# Payment using Razorpay UPI
class RazorpayUPIPayment(PaymentMethod):
    def __init__(self,upi_id):
        self.upi_id=upi_id

    def get_details(self):
        return "Razorpay UPI "+self.upi_id

    def pay(self,amount):
        print("Payment of Rs.",amount,"made using",self.get_details())
        return True

# Payment using Stripe card
class StripeCardPayment(PaymentMethod):
    def __init__(self,card_number):
        self.card_number=card_number

    def get_details(self):
        return "Stripe Card ending with "+self.card_number[-4:]

    def pay(self,amount):
        print("Payment of Rs.",amount,"made using",self.get_details())
        return True

# Payment using Stripe UPI
class StripeUPIPayment(PaymentMethod):
    def __init__(self,upi_id):
        self.upi_id=upi_id

    def get_details(self):
        return "Stripe UPI "+self.upi_id

    def pay(self,amount):
        print("Payment of Rs.",amount,"made using",self.get_details())
        return True

# Creates the selected payment object
class FactoryPaymentMethod(ABC):
    factory={}

    @classmethod
    def get_payment_object(cls,method_type,**kwargs):
        if method_type not in cls.factory:
            raise ValueError("Invalid payment method")
        return cls.factory[method_type](**kwargs)

class RazorpayFactory(FactoryPaymentMethod):
    factory={"card":RazorpayCardPayment,"upi":RazorpayUPIPayment}

class StripeFactory(FactoryPaymentMethod):
    factory={"card":StripeCardPayment,"upi":StripeUPIPayment}

# Base class for payment gateways
class Aggregator(ABC):
    def __init__(self,name,processing_fee,pay_factory):
        self.name=name
        self.processing_fee=processing_fee
        self.pay_factory=pay_factory

    @abstractmethod
    def call_get_payment_object(self,method_type,amount,**kwargs):
        pass

# Handles Razorpay payments and its fee
class RazorpayAggregator(Aggregator):
    def __init__(self):
        super().__init__("Razorpay",2.0,RazorpayFactory)

    def call_get_payment_object(self,method_type,amount,**kwargs):
        payment=self.pay_factory.get_payment_object(method_type,**kwargs)
        total=amount+(amount*self.processing_fee/100)
        print("Processing through Razorpay")
        print("Processing fee:",self.processing_fee,"%")
        return payment.pay(total)

# Handles Stripe payments and its fee
class StripeAggregator(Aggregator):
    def __init__(self):
        super().__init__("Stripe",2.9,StripeFactory)

    def call_get_payment_object(self,method_type,amount,**kwargs):
        payment=self.pay_factory.get_payment_object(method_type,**kwargs)
        total=amount+(amount*self.processing_fee/100)
        print("Processing through Stripe")
        print("Processing fee:",self.processing_fee,"%")
        return payment.pay(total)

# Selects the gateway entered by the user
class AggregatorFactory:
    factory={"stripe":StripeAggregator,"razorpay":RazorpayAggregator}

    @classmethod
    def register_aggregator(cls,name,aggregator_class):
        cls.factory[name.lower()]=aggregator_class

    @classmethod
    def get_aggregator_object(cls,name):
        name=name.lower()
        if name not in cls.factory:
            raise ValueError("Invalid aggregator")
        return cls.factory[name]()

# Takes details and starts the payment

def run_program():
    print("Multi-Gateway Payment Processing")
    try:
        gateway=input("Select gateway (stripe/razorpay): ").lower()
        aggregator=AggregatorFactory.get_aggregator_object(gateway)
        method=input("Select payment method (card/upi): ").lower()

        if method=="card":
            card_number=input("Enter card number: ")
            amount=float(input("Enter amount: "))
            result=aggregator.call_get_payment_object(method,amount,card_number=card_number)
        elif method=="upi":
            upi_id=input("Enter UPI ID: ")
            amount=float(input("Enter amount: "))
            result=aggregator.call_get_payment_object(method,amount,upi_id=upi_id)
        else:
            print("Invalid payment method")
            return

        if result:
            print("Payment successful")
        else:
            print("Payment failed")
    except ValueError as err:
        print("Error:",err)

if __name__=="__main__":
    run_program()
